In [10]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
from openai import OpenAI
import json

import utils.prompt_manager as prompt_manager
import utils.pdf_to_text as pdf_to_text
import utils.create_sanitized_folder as create_sanitized_folder
import utils.copy_pdf_to_folder as copy_pdf_to_folder
import model.model as model
import utils.incident_card_utils as incident_card_utils
from causal_graphviz.plot_conditions import draw_causal_graph
from utils.causal_postprocess import process_combined_text
from utils.causal_graph_interactive_pkg.causal_graph_interactive import (
    draw_causal_graph_interactive as draw_causal_graph_html,
)

# Your Open AI api
client = OpenAI()

# Paths
from pathlib import Path

folder = Path(r"runs\batch_api_test\batch_1\Sterigenics_Ontario_California_Ethylene_Oxide_Release_4_Injured")
hazard_consequence_json = "prompt\hazards_consequence.json"
conditions_json = "prompt\conditions.json"

model_name = "gpt-5.2"
reasoning_effort = "high" # "none" | "low" | "medium" | "high" | "xhigh"
verbosity="medium"          # "low" | "normal" | "high"

# identify_name, identify_incident, simplify_incident, identify_deviations_equipments, identify_hazard, 
# identify_condition, relate_hazards, relate_scenario, chain_events, chain_scenario, finalize_scenario
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
pdf_path = str(folder / "Tosco_Final_Report.pdf")
uploaded_pdf = client.files.create(
    file=open(pdf_path, "rb"),
    purpose="assistants",
)
print("Uploaded:", uploaded_pdf.id)

# 2. 构造 input（代替 messages）
input_messages = [
    {
        "role": "system",
        "content": [
            {
                "type": "input_text",
                "text": all_prompts["identify_incident"],
            }
        ],
    },
    {
        "role": "user",
        "content": [
            {
                "type": "input_text",
                "text": "The document is uploaded. Please perform the identify_incident task.",
            },
            {
                "type": "input_file",
                "file_id": uploaded_pdf.id, 
            },
        ],
    },
]

# 3. 调用 Responses API（没有 messages，只有 input）
response = client.responses.create(
    model=model_name,
    input=input_messages,
)

# 4. 拿到文本结果（官方推荐属性）
result_text = response.output_text
print(result_text)

# 5. 保存到当前 folder 下
output_path = folder / "identify_incident_output.txt"
with open(output_path, "w", encoding="utf-8") as f:
    f.write(result_text)

print(f"Saved output to: {output_path}")


FileNotFoundError: [Errno 2] No such file or directory: 'runs\\test_batch\\batch_1\\Removal_of_Hazardous_Material_from_Piping_Systems\\Tosco_Final_Report.pdf'

In [ ]:
identify_incident_output = folder / "identify_incident_output.txt"

with open(identify_incident_output, "r", encoding="utf-8") as f:
    identify_incident_output = f.read()

In [ ]:
from pathlib import Path

def render_prompt_only(prompt, variables=None):
    """
    Render a prompt using the same variable resolution logic as run_prompt,
    but without calling any model.
    """
    if variables and isinstance(variables, dict):
        resolved_vars = {}
        for k, v in variables.items():
            # Auto-read file content if v is a path
            if isinstance(v, (str, Path)) and Path(v).exists():
                text = Path(v).read_text(encoding="utf-8")
                resolved_vars[k] = text
            else:
                resolved_vars[k] = v

        prompt = prompt.format(**resolved_vars)

    print(prompt)
    return prompt

import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
prompt_text = all_prompts["chain_hazards"]

rendered = render_prompt_only(
    prompt=prompt_text,
    variables={"identify_hazard_consequence_output": folder/"identify_hazard_consequence_output.txt",
               "identify_incident_output": folder/"identify_incident_output.txt",
               "conditions_json": conditions_json,
               "chain_scenario_output": folder/"chain_scenario_output.txt"},
)

You are a professional process safety analyst.

INPUTS:
IDENTIFIED_HAZARD_CONSEQUENCE: Name: Confined explosion,  
Evidence: "The CSB has concluded that the initial explosion occurred inside a vertical above-ground storage tank that was being filled with Varnish Makers’ and Painters’ (VM&P) naphtha." "• The tank contained an ignitable vapor-air mixture in its head space."  
Explanation: An ignitable vapor-air mixture was present and ignited inside the enclosed tank headspace, producing an internal (confined) explosion.

Name: Fire ball,  
Evidence: "Witnesses heard the explosion and saw the fireball from several miles away."  
Explanation: The described “fireball” indicates rapid, intense combustion following the explosion, consistent with a fireball hazard consequence.

Name: Pool fire,  
Evidence: "Within moments, two more tanks ruptured and released their contents into the rapidly escalating fire that was concentrated inside the earthen spill containment area surrounding the tank fa

In [50]:
# identify_hazard_consequence
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
identify_hazard_consequence = model.run_prompt(
    prompt=all_prompts["identify_hazard_consequence"],
    variables={"hazards_consequence_json": hazard_consequence_json,
               "identify_incident_output": (folder / "identify_incident_output.txt")},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="identify_hazard_consequence",
    prev_prompt= None,
    prev_output = None
)

In [51]:
# identify_condition
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
identify_condition_outputs = model.run_prompt(
    prompt=all_prompts["identify_condition"],
    variables={
               "identify_hazard_consequence_output": folder/"identify_hazard_consequence_output.txt",
               "identify_incident_output": folder / "identify_incident_output.txt",
               "conditions_json": conditions_json
               },
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="identify_condition",
    prev_prompt= None,
    prev_output = None
)

In [12]:
# identify_evidence
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
identify_evidence_outputs = model.run_prompt(
    prompt=all_prompts["identify_evidence"],
    variables={
               "identify_condition_output": folder/"identify_condition_output.txt",
               "identify_incident_output": folder / "identify_incident_output.txt",
               "conditions_json": conditions_json
               },
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="identify_evidence",
    prev_prompt= None,
    prev_output = None
)

In [13]:
# chain_events
chain_events_outputs = model.run_prompt(
    prompt=all_prompts["chain_events"],
    variables={"identify_evidence_output": folder/"identify_evidence_output.txt"},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="chain_events",
    prev_prompt= None,
    prev_output = None
)

In [14]:
# delete the repetitive events
delete_repetitive_events = model.run_prompt(
    prompt=all_prompts["delete_repetitive_events"],
    variables={"chain_events_output": folder/"chain_events_output.txt",
               "conditions_json": conditions_json},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="delete_repetitive_events",
    prev_prompt= None,
    prev_output = None
)

In [15]:
# update the results of chain event
update_chain_events = model.run_prompt(
    prompt=all_prompts["update_chain_events"],
    variables={"chain_events_output": folder/"chain_events_output.txt",
               "delete_repetitive_events_output": folder/"delete_repetitive_events_output.txt"},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="update_chain_events",
    prev_prompt= None,
    prev_output = None
)

In [16]:
# relationship of conditions and events
identify_relationship = model.run_prompt(
    prompt=all_prompts["identify_relationship"],
    variables={"update_chain_events_output": folder/"update_chain_events_output.txt",
               "identify_condition_output": folder/"identify_condition_output.txt",
               "conditions_json": conditions_json},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="identify_relationship",
    prev_prompt= None,
    prev_output = None
)

In [17]:
# chain_conditions and events
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
chain_conditions_events = model.run_prompt(
    prompt=all_prompts["chain_conditions_events"],
    variables={"identify_relationship_output": folder/"identify_relationship_output.txt",
               "identify_hazard_consequence_output": folder/"identify_hazard_consequence_output.txt",},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="chain_conditions_events",
    prev_prompt= None,
    prev_output = None
)

In [18]:
# chain_scenario
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
chain_scenario_outputs = model.run_prompt(
    prompt=all_prompts["chain_scenario"],
    variables={"identify_hazard_consequence_output": folder/"identify_hazard_consequence_output.txt",
               "update_chain_events_output": folder/"update_chain_events_output.txt",
               "conditions_json": conditions_json, },
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="chain_scenario",
    prev_prompt=None,
    prev_output = None
)

In [19]:
# chain_hazards
import utils.prompt_manager as prompt_manager
all_prompts = prompt_manager.prompts.load_all()
chain_hazards_outputs = model.run_prompt(
    prompt=all_prompts["chain_hazards"],
    variables={"identify_hazard_consequence_output": folder/"identify_hazard_consequence_output.txt",
               "identify_incident_output": folder/"identify_incident_output.txt",
               "conditions_json": conditions_json,
               "chain_scenario_output": folder/"chain_scenario_output.txt"},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="chain_hazards",
    prev_prompt= None,
    prev_output = None
)

In [ ]:
final_check_outputs = model.run_prompt(
    prompt=all_prompts["final_check"],
    variables={"combined_text_output": combined_text_updated,
               "hazards_consequence_json": hazard_consequence_json,
               "conditions_json": conditions_json},
    output_dir=folder,
    model_name=model_name,
    reasoning_effort=reasoning_effort,
    verbosity=verbosity,
    prompt_key="final_check",
    prev_prompt= None,
    prev_output = None
)

import winsound
winsound.Beep(1000, 500)

In [ ]:

graph_png = draw_causal_graph(
    chain_lines=(folder / "final_check_output.txt").read_text(encoding="utf-8"),
    # chain_lines = combined_text,
    conditions="prompt/conditions.json",
    hazards = "prompt/hazards_consequence.json",
    save_path=folder / "causal_graph.png"
)

In [3]:
# %load_ext autoreload
# %autoreload 2

# import sys
# from pathlib import Path
# from openai import OpenAI
# import json

# import utils.prompt_manager as prompt_manager
# import utils.pdf_to_text as pdf_to_text
# import utils.create_sanitized_folder as create_sanitized_folder
# import utils.copy_pdf_to_folder as copy_pdf_to_folder
# import model.model as model
# import utils.incident_card_utils as incident_card_utils
# from causal_graphviz.plot_conditions import draw_causal_graph

# # Your Open AI api
# client = OpenAI()

# # Paths
# from pathlib import Path

# folder = Path(r"runs\test_batch\batch_2\MFG_Chemical_Inc_Toxic_Chemical_Vapor_Cloud_Release")
# hazard_consequence_json = "prompt\hazards_consequence.json"
# conditions_json = "prompt\conditions.json"

# model_name = "gpt-5.2"
# reasoning_effort = "high" # "none" | "low" | "medium" | "high" | "xhigh"
# verbosity="high"          # "low" | "normal" | "high"

# # identify_name, identify_incident, simplify_incident, identify_deviations_equipments, identify_hazard, 
# # identify_condition, relate_hazards, relate_scenario, chain_events, chain_scenario, finalize_scenario
# import utils.prompt_manager as prompt_manager
# all_prompts = prompt_manager.prompts.load_all()

from causal_graph_interactive import draw_causal_graph_interactive
from pathlib import Path
chain_text = (folder / "final_check_output.txt").read_text(encoding="utf-8")
graph_html = draw_causal_graph_interactive(
    chain_lines=chain_text,
    conditions="prompt/conditions.json",
    hazards="prompt/hazards_consequence.json",
    save_path=folder / "causal_graph.html"
)

In [63]:
# incident_card_utils.incident_card_utils(
#     identify_incident_prompt = all_prompts["identify_incident"],
#     identify_hazard_consequence_prompt = all_prompts["identify_hazard_consequence"],
#     identify_condition_prompt = all_prompts["identify_condition"],
#     identify_evidence_prompt = all_prompts["identify_evidence"],
#     identify_relationship_prompt = all_prompts["identify_relationship"],
#     chain_events_prompt = all_prompts["chain_events"],
#     chain_conditions_events_prompt = all_prompts["chain_conditions_events"],
#     chain_scenario_prompt = all_prompts["chain_scenario"],
#     chain_hazards_prompt = all_prompts["chain_hazards"],

#     identify_incident_output = folder / "identify_incident_output.txt",
#     identify_hazard_consequence_output = folder / "identify_hazard_consequence_output.txt",
#     identify_condition_output = folder / "identify_condition_output.txt",
#     identify_evidence_output = folder / "identify_evidence_output.txt",
#     identify_relationship_output = folder / "identify_relationship_output.txt",
#     chain_events_output = folder / "chain_events_output.txt",
#     chain_conditions_events_output = folder / "chain_conditions_events_output.txt",
#     chain_scenario_output = folder / "chain_scenario_output.txt",
#     chain_hazards_output = folder / "chain_hazards_output.txt",

#     hazard_consequence_json = hazard_consequence_json,
#     conditions_json = conditions_json,

#     graph_png = folder / "causal_graph.png",

#     md_path=folder,
#     file_name=f"results.md"
# )

In [66]:
from datetime import datetime
from pathlib import Path

from docx.shared import Pt
from docx.enum.text import WD_PARAGRAPH_ALIGNMENT
import utils.incident_card_to_word as incident_card_to_word

incident_card_to_word.incident_card_to_word(
    identify_incident_prompt = all_prompts["identify_incident"],
    identify_hazard_consequence_prompt = all_prompts["identify_hazard_consequence"],
    identify_condition_prompt = all_prompts["identify_condition"],
    identify_evidence_prompt = all_prompts["identify_evidence"],
    chain_events_prompt = all_prompts["chain_events"],
    delete_repetitive_events_prompt = all_prompts["delete_repetitive_events"],
    update_chain_events_prompt = all_prompts["update_chain_events"],
    identify_relationship_prompt = all_prompts["identify_relationship"],
    chain_conditions_events_prompt = all_prompts["chain_conditions_events"],
    chain_scenario_prompt = all_prompts["chain_scenario"],
    chain_hazards_prompt = all_prompts["chain_hazards"],
    final_check_prompt = all_prompts["final_check"],

    identify_incident_output = folder / "identify_incident_output.txt",
    identify_hazard_consequence_output = folder / "identify_hazard_consequence_output.txt",
    identify_condition_output = folder / "identify_condition_output.txt",
    identify_evidence_output = folder / "identify_evidence_output.txt",
    chain_events_output = folder / "chain_events_output.txt",
    delete_repetitive_events_output = folder / "delete_repetitive_events_output.txt",
    update_chain_events_output = folder / "update_chain_events_output.txt",
    identify_relationship_output = folder / "identify_relationship_output.txt",
    chain_conditions_events_output = folder / "chain_conditions_events_output.txt",
    chain_scenario_output = folder / "chain_scenario_output.txt",
    chain_hazards_output = folder / "chain_hazards_output.txt",
    final_check_output = folder / "final_check_output.txt",

    hazard_consequence_json = hazard_consequence_json,
    conditions_json = conditions_json,

    graph_png = folder / "causal_graph.png",
    output_docx_path = folder / "incident_card_report.docx"
)

import winsound
winsound.Beep(1000, 500)

In [2]:
chain_scenario_text = (folder / "chain_scenario_output.txt").read_text(encoding="utf-8")
chain_hazards_text = (folder / "chain_hazards_output.txt").read_text(encoding="utf-8")
chain_conditions_events_text = (folder / "chain_conditions_events_output.txt").read_text(encoding="utf-8")
combined_text = (
    chain_scenario_text
    + "\n\n"
    + chain_hazards_text
    + "\n\n"
    + chain_conditions_events_text
)

final_prompt = model.run_prompt(
    prompt=all_prompts["final_check"],
    variables={"combined_text_output": combined_text,
               "hazards_consequence_json": hazard_consequence_json,
               "conditions_json": conditions_json},
    preview_only=True,     # ✅ 不跑 API
    print_prompt=True,     # ✅ 打印出来
)


========== FINAL PROMPT ==========

You are a professional process safety analyst.

INPUT (JSON):

COMBINED_TEXT: "VM&P naphtha -> inside tanks\ninside tanks -> Chemical combustible <Confined explosion 1>\nVM&P naphtha -> into 15,000 gallon tank\ninto 15,000 gallon tank -> Liquid <Confined explosion 1>\nignitable vapor-air mixture -> tank head space\ntank head space -> Vapor <Confined explosion 1>\nignitable vapor-air mixture -> tank head space\ntank head space -> Pre-mixture <Confined explosion 1>\ncreated a spark -> occurred inside storage tank\noccurred inside storage tank -> Instant Ignition <Confined explosion 1>\nvertical above-ground storage tank -> occurred inside storage tank\noccurred inside storage tank -> High confinement <Confined explosion 1>\ntank head space -> ignitable vapor-air mixture\nignitable vapor-air mixture -> Vapor cloud formation <Confined explosion 1>\nflammable liquid -> into the spill containment\ninto the spill containment -> Chemical combustible <Confine

In [ ]:
from utils.causal_postprocess import process_combined_text

chain_scenario_text = (folder / "chain_scenario_output.txt").read_text(encoding="utf-8")
chain_hazards_text = (folder / "chain_hazards_output.txt").read_text(encoding="utf-8")
chain_conditions_events_text = (folder / "chain_conditions_events_output.txt").read_text(encoding="utf-8")
combined_text = (
    chain_scenario_text
    + "\n\n"
    + chain_hazards_text
    + "\n\n"
    + chain_conditions_events_text
)
updated = process_combined_text(
    combined_text=combined_text,
    conditions=conditions_json,                 # 你的 conditions_json dict
    hazard_consequence=hazard_consequence_json, # 你的 hazard list
    dedupe_edges=True,                    # 是否去重
)

print(updated)

VM&P naphtha flammable liquid -> In vertical above-ground storage tank
In vertical above-ground storage tank -> NFPA Class IB flammable liquid
NFPA Class IB flammable liquid -> Chemical combustible <Confined explosion 1>
VM&P naphtha liquid transfer -> In tanker-trailer compartment
In tanker-trailer compartment -> Into 15,000 gallon storage tank
Into 15,000 gallon storage tank -> Explicit bulk liquid transfer
Explicit bulk liquid transfer -> Liquid <Confined explosion 1>
VM&P naphtha vapors with air -> Inside tanks
Inside tanks -> Can produce ignitable mixtures
Can produce ignitable mixtures -> Vapor <Confined explosion 1>
Ignitable vapor-air mixture present -> In tank head space
In tank head space -> Fuel and air already mixed
Fuel and air already mixed -> Pre-mixture <Confined explosion 1>
Float linkage likely separated -> Created a spark
Created a spark -> During filling
During filling -> At gauging system float
At gauging system float -> Soon after transfer started
Soon after trans

Saved: hazard_consequence_flow.png
